# Physical inactivity and diabetes across US counties

**STA 556 Final Project** · *Your name*

**Question.** Across US counties, how strongly is the prevalence of physical inactivity associated with diabetes prevalence, after accounting for obesity and smoking?

Run this notebook from the repository root. The first run downloads the data from the CDC API and caches it in `data/raw/`; later runs read the cached file.

## 1. Question and data

- **Data source:** CDC PLACES: Local Data for Better Health, County Data, 2025 release (dataset `swc5-untb`), accessed through the Socrata (SODA) API at `data.cdc.gov`.
- **Unit of observation:** a US county (50 states + DC).
- **Variables (age-adjusted prevalence, % of adults):**
  - `diabetes` (DIABETES): diagnosed diabetes, *response*
  - `lpa` (LPA): no leisure-time physical activity, *main predictor*
  - `obesity` (OBESITY): obesity
  - `csmoking` (CSMOKING): current smoking
- **Main limitations:** *TODO: summarize here (model-based estimates, ecological data, observational design). See Section 7.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.fetch import fetch_places, MEASURES
from src.clean import validate_raw, tidy_places, missingness_report, drop_incomplete
from src.analysis import (describe, design_matrix, ols_fit, coef_table,
                          bootstrap_coefs, percentile_ci)

RAW_PATH = Path("data/raw/places_county_2025.csv")
PROCESSED_PATH = Path("data/processed/county_analysis.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

Y = "diabetes"
X_COLS = ["lpa", "obesity", "csmoking"]
SEED = 556
N_BOOT = 2000

## 2. Data workflow

### 2a. Download (or load cached) raw data
The raw file is saved exactly as the API returned it and never edited by hand. On the first download, `fetch_places` also writes `data/raw/places_county_2025_provenance.json`, recording the endpoint, query parameters, and retrieval date.

In [ ]:
raw = fetch_places(MEASURES, RAW_PATH)
print(raw.shape)
raw.head()

### 2b. Validate the raw data

In [ ]:
facts = validate_raw(raw, MEASURES)
facts

### 2c. Reshape: long to wide
The API returns one row per county per measure. We pivot so each county is one row with one column per measure, keeping only the 50 states and DC.

In [ ]:
wide = tidy_places(raw, MEASURES)
print(wide.shape)
wide.head()

### 2d. Missingness

In [ ]:
missingness_report(wide, [Y] + X_COLS)

In [ ]:
county, n_dropped = drop_incomplete(wide, [Y] + X_COLS)
print(f"Dropped {n_dropped} counties with a missing analysis variable; {len(county)} remain.")
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
county.to_csv(PROCESSED_PATH, index=False)

*TODO: explain the missingness decision (how many counties, why complete-case analysis is reasonable here).*

## 3. Descriptive summaries

In [ ]:
describe(county, [Y] + X_COLS)

In [ ]:
county[[Y] + X_COLS].corr().round(2)

## 4. Exploratory plots

In [ ]:
labels = {"diabetes": "Diabetes", "lpa": "Physical inactivity",
          "obesity": "Obesity", "csmoking": "Current smoking"}
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, [Y] + X_COLS):
    ax.hist(county[col], bins=40, color="#4C72B0", edgecolor="white")
    ax.set_title(labels[col])
    ax.set_xlabel("Prevalence (%)")
axes[0].set_ylabel("Counties")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_histograms.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, col in zip(axes, X_COLS):
    ax.scatter(county[col], county[Y], s=6, alpha=0.3, color="#4C72B0")
    ax.set_xlabel(f"{labels[col]} (%)")
axes[0].set_ylabel("Diabetes (%)")
fig.suptitle("Diabetes prevalence against each predictor (one point per county)")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_scatter.png", dpi=150)

*TODO: describe what the plots show (shapes, skew, outliers, strength of each relationship, collinearity among predictors).*

## 5. Least squares with NumPy

Model: $\text{diabetes}_i = \beta_0 + \beta_1\,\text{lpa}_i + \beta_2\,\text{obesity}_i + \beta_3\,\text{csmoking}_i + \varepsilon_i$

Coefficients are computed with `np.linalg.lstsq`, and standard errors with `np.linalg.solve` instead of an explicit matrix inverse.

In [ ]:
X = design_matrix(county, X_COLS)
y = county[Y].to_numpy()
fit = ols_fit(X, y)
print(f"n = {len(y)}, rank(X) = {fit['rank']}, condition number = {fit['cond']:.1f}, R^2 = {fit['r2']:.3f}")
coef_table(fit, X_COLS)

### Diagnostics

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(fit["fitted"], fit["resid"], s=6, alpha=0.3, color="#4C72B0")
axes[0].axhline(0, color="black", lw=1)
axes[0].set(xlabel="Fitted diabetes (%)", ylabel="Residual", title="Residuals vs fitted")
stats.probplot(fit["resid"], dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q plot of residuals")
fig.tight_layout()
fig.savefig(FIG_DIR / "diagnostics.png", dpi=150)

*TODO: interpret R², the condition number (are the predictors collinear?), and the residual plots (curvature, unequal spread, heavy tails). If spread is unequal, the classical SEs are unreliable, which motivates the bootstrap below.*

## 6. Bootstrap uncertainty

**Mechanism.** We resample counties with replacement (a case, or pairs, bootstrap): each bootstrap sample draws $n$ rows of $(x_i, y_i)$ from the data, and we refit the least-squares model. The spread of the refitted coefficients estimates their sampling variability without assuming normal, constant-variance errors.

In [ ]:
rng = np.random.default_rng(SEED)
boot = bootstrap_coefs(X, y, n_boot=N_BOOT, rng=rng)
ci = percentile_ci(boot)
results = coef_table(fit, X_COLS)
results["boot_se"] = boot.std(axis=0, ddof=1).round(4)
results["ci_low"], results["ci_high"] = ci[:, 0].round(4), ci[:, 1].round(4)
results

In [ ]:
j = 1  # column of X for physical inactivity
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(boot[:, j], bins=50, color="#4C72B0", edgecolor="white")
for v in ci[j]:
    ax.axvline(v, color="black", ls="--", lw=1)
ax.set(xlabel="Bootstrap estimate of the physical-inactivity coefficient",
       ylabel="Count", title=f"Bootstrap distribution (B = {N_BOOT})")
fig.tight_layout()
fig.savefig(FIG_DIR / "bootstrap_lpa.png", dpi=150)

*TODO: interpret the 95% CI for the physical-inactivity coefficient in plain words (e.g., 'holding obesity and smoking fixed, counties with 1 percentage point more inactivity have about … points higher diabetes prevalence'). Compare bootstrap SEs to the classical SEs. Note that rerunning with a different seed changes the CI endpoints slightly (Monte Carlo variability) and that B = 2000 keeps that small.*

## Communication figure

In [ ]:
b_lpa, (lo, hi) = fit["beta"][1], ci[1]
slope, intercept = np.polyfit(county["lpa"], county[Y], 1)
xs = np.linspace(county["lpa"].min(), county["lpa"].max(), 100)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(county["lpa"], county[Y], s=8, alpha=0.25, color="#4C72B0", label="County")
ax.plot(xs, intercept + slope * xs, color="#C44E52", lw=2.5, label="Least-squares line")
ax.set_xlabel("Adults with no leisure-time physical activity (%)", fontsize=11)
ax.set_ylabel("Adults with diagnosed diabetes (%)", fontsize=11)
ax.set_title("Less active counties have higher diabetes prevalence", fontsize=13, loc="left")
ax.text(0.01, 0.97,
        f"After adjusting for obesity and smoking: +{b_lpa:.2f} points per 1-point rise in inactivity "
        f"(95% bootstrap CI {lo:.2f} to {hi:.2f})",
        transform=ax.transAxes, va="top", fontsize=9, color="#333333")
ax.text(0.99, 0.01, "Source: CDC PLACES 2025 county release (age-adjusted estimates)",
        transform=ax.transAxes, ha="right", fontsize=8, color="#666666")
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.legend(loc="lower right", bbox_to_anchor=(1, 0.05), frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "communication_figure.png", dpi=200)

## 7. Limitations

*TODO: write 1-2 short paragraphs. Points to cover:*
- *PLACES values are model-based small-area estimates (from BRFSS survey data and census covariates), not direct county counts; they carry their own uncertainty, which this analysis ignores.*
- *Because the estimates are partly built from the same county demographics, associations between measures may be inflated.*
- *Ecological data: county-level associations do not describe individuals.*
- *Observational design: no causal conclusions; income, age structure, healthcare access, and other confounders are omitted.*
- *Counties are not independent (neighbouring counties are similar), so even the bootstrap understates uncertainty somewhat.*

## 8. Conclusion

*TODO: answer the question in 2-3 sentences.*